# RealVisXL V4.0 Separate Pipeline Studio
Uses sequential CPU offloading to load txt2img and img2img pipelines independently.

In [ ]:
!pip install -q diffusers transformers accelerate torch torchvision gradio

import torch
from diffusers import AutoPipelineForImage2Image, AutoPipelineForText2Image
import gradio as gr

def get_txt2img():
    pipe = AutoPipelineForText2Image.from_pretrained("SG161222/RealVisXL_V4.0", torch_dtype=torch.float16, use_safetensors=True)
    pipe.enable_sequential_cpu_offload()
    return pipe

def get_img2img():
    pipe = AutoPipelineForImage2Image.from_pretrained("SG161222/RealVisXL_V4.0", torch_dtype=torch.float16, use_safetensors=True)
    pipe.enable_sequential_cpu_offload()
    return pipe

pipe_t2i = get_txt2img()
pipe_i2i = get_img2img()

def generate(prompt, neg, steps):
    return pipe_t2i(prompt=prompt, negative_prompt=neg, num_inference_steps=int(steps), guidance_scale=7.5).images[0]

def edit(image, prompt, neg, strength, steps):
    return pipe_i2i(prompt=prompt, negative_prompt=neg, image=image, strength=float(strength), num_inference_steps=int(steps), guidance_scale=7.5).images[0]

with gr.Blocks() as demo:
    gr.Markdown("# RealVisXL V4.0 Separate Studio")
    with gr.Tab("Generate"): 
        gen_out = gr.Image(); gen_btn = gr.Button("Generate")
        gen_btn.click(generate, [gr.Textbox(label="Prompt"), gr.Textbox(label="Neg"), gr.Slider(10, 50, 25)], gen_out)
    with gr.Tab("Edit"): 
        edit_out = gr.Image(); edit_btn = gr.Button("Edit")
        edit_btn.click(edit, [gr.Image(type="pil"), gr.Textbox(label="Prompt"), gr.Textbox(label="Neg"), gr.Slider(0.1, 0.9, 0.5), gr.Slider(10, 50, 25)], edit_out)

demo.launch(share=True)